# E3 — 09 : audit des tenseurs et des cibles avant entraînement

**Lecture seule** : aucune modification des données initiales. Exécuter depuis le dossier principal du projet TalTech, après les notebooks E3_06, E3_07ABCD et E3_08. L'audit vérifie les correspondances entre clés, indices, représentations, cibles et normalisations, puis exporte un rapport. Il ne constitue pas une preuve générale d'absence de fuite de données.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
REP = ROOT/'outputs/E3_aligned_representations'
COMMON = ROOT/'outputs/E3_common_evaluation'
OUT = ROOT/'outputs/E3_pretraining_audit'
OUT.mkdir(parents=True,exist_ok=True)
HORIZONS=[15,30,60,120]
SPLITS=['train','validation','test']
KEYS=['uuid','timestamp','split']
VARS=['temperature','relative_humidity','co2']
assert REP.exists(), f'Missing {REP}'
assert COMMON.exists(), f'Missing {COMMON}'
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
for col in ['timestamp']:
    common[col]=pd.to_datetime(common[col],utc=True)
assert not common.duplicated(KEYS).any()
print('Common keys:',len(common),'by split:',common.groupby('split').size().to_dict())

In [ ]:
def load_metadata(h,s):
    p=REP/f'h{h:03d}'/f'metadata_{s}.csv'
    assert p.exists(),p
    d=pd.read_csv(p)
    d['timestamp']=pd.to_datetime(d['timestamp'],utc=True)
    d['target_timestamp']=pd.to_datetime(d['target_timestamp'],utc=True)
    assert not d.duplicated(KEYS).any(),f'duplicate keys {h} {s}'
    assert d['split'].eq(s).all()
    assert (d['target_timestamp']-d['timestamp']).eq(pd.Timedelta(minutes=h)).all()
    return d

def subset(h,s):
    d=load_metadata(h,s)
    idx=np.load(COMMON/f'indices_h{h:03d}_{s}.npy')
    assert idx.ndim==1 and np.issubdtype(idx.dtype,np.integer)
    assert len(idx)==len(set(idx.tolist())) and (idx>=0).all() and (idx<len(d)).all()
    selected=d.iloc[idx].reset_index(drop=True)
    ref=common.loc[common.split.eq(s),KEYS].sort_values(KEYS).reset_index(drop=True)
    got=selected[KEYS].sort_values(KEYS).reset_index(drop=True)
    pd.testing.assert_frame_equal(got,ref,check_dtype=False)
    return d,idx,selected

rows=[]; refs={}; selected_targets={}
for h in HORIZONS:
    for s in SPLITS:
        base=REP/f'h{h:03d}'
        d,idx,selected=subset(h,s)
        xseq=np.load(base/f'X_sequence_{s}_unscaled.npy',mmap_mode='r')
        xtab=np.load(base/f'X_tabular_{s}_unscaled.npy',mmap_mode='r')
        xgraph=np.load(base/f'X_graph_{s}_unscaled.npy',mmap_mode='r')
        yseq=np.load(base/f'y_sequence_{s}.npy',mmap_mode='r')
        ytab=np.load(base/f'y_tabular_{s}.npy',mmap_mode='r')
        ygraph=np.load(base/f'y_graph_{s}.npy',mmap_mode='r')
        assert xseq.shape==(len(d),5,3),(h,s,xseq.shape,len(d))
        assert xtab.shape==(len(d),15)
        assert xgraph.shape==(len(d)//7,5,7,3)
        assert yseq.shape==(len(d),) and ytab.shape==(len(d),) and ygraph.shape==(len(d)//7,7)
        assert np.array_equal(np.asarray(xtab),np.asarray(xseq).reshape(-1,15))
        assert np.array_equal(np.asarray(xgraph).transpose(0,2,1,3).reshape(-1,5,3),np.asarray(xseq))
        assert np.array_equal(np.asarray(yseq),np.asarray(ytab))
        assert np.array_equal(np.asarray(ygraph).reshape(-1),np.asarray(yseq))
        target_col=f'target_temperature_t_plus_{h}'
        if target_col in d.columns:
            assert np.allclose(np.asarray(yseq),d[target_col].to_numpy(dtype=float),rtol=0,atol=1e-5)
        # Validate histories are identical for the same device-time keys across horizons.
        x_selected=np.asarray(xseq[idx]).copy()
        keys=list(map(tuple,selected[KEYS].itertuples(index=False,name=None)))
        if s not in refs:
            refs[s]={k:v for k,v in zip(keys,x_selected)}
        else:
            for k,v in zip(keys,x_selected):
                assert np.array_equal(refs[s][k],v),f'history differs at {h}, {s}, {k}'
        selected_targets[(h,s)]=np.asarray(yseq[idx]).copy()
        rows.append(dict(horizon_min=h,split=s,all_samples=len(d),common_samples=len(idx),snapshots=len(d)//7,history_identical=True,targets_aligned=True,representation_consistent=True))
        print(f'OK h={h} {s}: {len(idx)} common / {len(d)} total')
report=pd.DataFrame(rows)
report.to_csv(OUT/'01_pretraining_tensor_audit.csv',index=False)
print(report.to_string(index=False))

In [ ]:
# Verify scaling using the EXACT scaler parameters saved by E3_07ABCD.
# Do not recompute mean/std from float32 arrays: rounding and accumulation
# may differ from sklearn StandardScaler and cause false mismatches.
scale_rows=[]
for h in HORIZONS:
    base=REP/f'h{h:03d}'
    params=pd.read_csv(base/'02_scaler_train_only.csv')
    assert list(params['variable']) == ['temperature','relative_humidity','co2']
    mean=params['train_mean'].to_numpy(dtype=np.float64)
    std=params['train_scale'].to_numpy(dtype=np.float64)
    assert np.isfinite(mean).all() and np.isfinite(std).all() and (std>0).all()
    # Independently check that the saved scaler parameters describe training data.
    train=np.load(base/'X_sequence_train_unscaled.npy',mmap_mode='r')
    train64=np.asarray(train,dtype=np.float64).reshape(-1,3)
    mean_train=train64.mean(axis=0)
    std_train=train64.std(axis=0)
    train_mean_error=float(np.max(np.abs(mean_train-mean)))
    train_std_error=float(np.max(np.abs(std_train-std)))
    assert np.allclose(mean_train,mean,rtol=1e-6,atol=1e-5),f'Train mean mismatch h={h}: {train_mean_error}'
    assert np.allclose(std_train,std,rtol=1e-6,atol=1e-5),f'Train scale mismatch h={h}: {train_std_error}'
    for s in SPLITS:
        xx=np.load(base/f'X_sequence_{s}_unscaled.npy',mmap_mode='r')
        zz=np.load(base/f'X_sequence_{s}_scaled.npy',mmap_mode='r')
        # sklearn's transform works with float32 input and float64 scaler
        # statistics; the expected result is stored in float32.
        expected=((np.asarray(xx,dtype=np.float64)-mean)/std).astype(np.float32)
        max_abs=float(np.max(np.abs(expected-np.asarray(zz))))
        assert np.allclose(expected,zz,rtol=1e-5,atol=1e-5),f'Scaler mismatch h={h} {s}: {max_abs}'
        scale_rows.append(dict(horizon_min=h,split=s,max_abs_error=max_abs,train_mean_error=train_mean_error,train_std_error=train_std_error,scaler_source='saved_training_only_scaler'))
scales=pd.DataFrame(scale_rows)
scales.to_csv(OUT/'02_scaling_audit.csv',index=False)
print(scales.to_string(index=False))


In [ ]:
# Optional target verification against the original cleaned TalTech time series.
source=ROOT/'outputs/05_ml_dataset/06_ai_ready_complete_case.csv'
assert source.exists(),source
raw=pd.read_csv(source,usecols=['uuid','timestamp','temperature'])
raw['timestamp']=pd.to_datetime(raw['timestamp'],utc=True)
assert not raw.duplicated(['uuid','timestamp']).any()
lookup=raw.set_index(['uuid','timestamp'])['temperature']
checks=[]
for h in HORIZONS:
    for s in SPLITS:
        _,idx,selected=subset(h,s)
        future=pd.MultiIndex.from_frame(selected[['uuid','target_timestamp']].rename(columns={'target_timestamp':'timestamp'}))
        actual=lookup.reindex(future).to_numpy(dtype=float)
        predicted_target=selected_targets[(h,s)]
        assert np.isfinite(actual).all(),f'Missing source targets h={h}, {s}'
        max_abs=float(np.max(np.abs(actual-predicted_target)))
        assert np.allclose(actual,predicted_target,rtol=0,atol=1e-5),f'Target mismatch {h}, {s}, {max_abs}'
        checks.append(dict(horizon_min=h,split=s,common_samples=len(idx),max_abs_target_difference=max_abs))
pd.DataFrame(checks).to_csv(OUT/'03_source_target_audit.csv',index=False)
print('PASS: target values match the original time series at t+h')
print('Saved reports to:',OUT)

## Interprétation et limites

- Tous les tests doivent se terminer sans `AssertionError` avant les entraînements.
- L'audit confirme les clés, dimensions, égalité des historiques, cibles, transformation des scalers et valeurs des cibles sources.
- Il **ne vérifie pas** la provenance des arêtes du GCN, le déterminisme de l'entraînement ni l'absence de toute fuite temporelle dans d'autres scripts.
- Les modèles doivent être entraînés sur les **ensembles complets propres à chaque horizon**, et évalués avec les indices communs. Une analyse de sensibilité avec entraînement sur l'intersection commune peut être envisagée séparément.
- Envoyez les fichiers `01_pretraining_tensor_audit.csv`, `02_scaling_audit.csv`, `03_source_target_audit.csv` et tout message d'erreur éventuel.